# Honest Full-Precision Eval — Path A (Unsloth match training stack)

**Цель**: устранить inference-artifact в сравнении base vs GSPO vs KTO. Все три модели прогоняются через **Unsloth FastLanguageModel** (как в training notebook) с identical decoding protocol. Phase 0a: programmatic correctness only (fast_mode=True); Phase 0b — Cerebras judge поверх saved completions, отдельным async скриптом.

**Compute**: Colab A100 40GB. Estimated:
- С `causal-conv1d`: ~30 tok/s × 2048 max × 143 problems × 3 stages ≈ **5h**.
- Без `conv1d` (fla one): ~12 tok/s × 2048 × 143 × 3 ≈ **12h** или 50 stratified subset за **4h**.

**Output**: `evaluation/reports/honest_full_precision_phase0a_YYYYMMDD.json` (repo + Drive mirror).

**Стек (verified 2026-05-05)**: torch 2.10.0+cu128, transformers 5.5.0, trl 0.24.0, datasets 4.3.0, unsloth 2026.5.1, fla 0.5.0+. ALL strict-pinned под Unsloth constraints.

**Структура**:
1. **Setup** — install verified stack → **RESTART RUNTIME** → re-run cell to load imports
2. **DECODING_CONFIG** — single-protocol для трёх моделей (num_predict=2048, enable_thinking=True, temperature=0.0)
3. **Load eval dataset** — 143 calc problems (numeric + latex_boxed)
4. **Per-model inference** — base / +GSPO / +KTO via FastLanguageModel + PEFT merge
5. **Eval loop** — fast_mode=True (programmatic correctness, no Cerebras), per-problem timing для первых 3 problems
6. **Run + save** — JSON report со всеми completions для Phase 0b later


In [ ]:
# Cell 1: Setup (FINAL — empirically verified Unsloth-compatible stack)
# First run: install pinned versions → RESTART RUNTIME → re-run this cell.
# Sentinel /content/.install_done_v4 gates install; after restart imports load.
import subprocess, os

# ─── GPU check (need ≥24GB for 9B bf16) ────────────────────
gpu_info = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']
).decode().strip()
print(f'GPU: {gpu_info}')
gpu_memory_mib = int(gpu_info.split(',')[1].strip().split()[0])
assert gpu_memory_mib >= 24_000, (
    f'Insufficient VRAM for 9B bf16: {gpu_memory_mib} MiB. Need >=24GB.'
)
print(f'VRAM check passed: {gpu_memory_mib} MiB ({gpu_memory_mib/1024:.1f} GiB)')

# ─── Drive mount + repo clone (idempotent) ────────────────
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/MITS'):
    !git clone https://github.com/Siesher/MIST.git /content/MITS
%cd /content/MITS
!git checkout 019-ns-vstar-dpo && git pull origin 019-ns-vstar-dpo

# ─── Install (FIRST RUN ONLY — gated by sentinel v4) ──────
# Empirically verified stack — все версии strict-pinned под Unsloth 2026.5.1
# constraint set: transformers<=5.5.0, torch<2.11.0, trl<=0.24.0, datasets<4.4.0
SENTINEL = '/content/.install_done_v4'
if not os.path.exists(SENTINEL):
    print('=== Installing verified stack (Unsloth + transformers 5.5.0 + torch 2.10+cu128) ===')

    # Step 1: pytorch ecosystem на torch 2.10+cu128 (Unsloth max + matches system nvcc 12.8)
    # --extra-index-url нужен для cu128 wheels (на PyPI mirror)
    print('Step 1: pinning torch 2.10+cu128...')
    !pip install -q --force-reinstall \
        "torch==2.10.0" "torchvision" "torchaudio" \
        --extra-index-url https://download.pytorch.org/whl/cu128

    # Step 2: HF stack — strict pin к Unsloth-compatible versions
    # transformers 5.5.0 — последняя в Unsloth bound, имеет qwen3_5 architecture
    print('Step 2: HF stack (transformers 5.5.0, trl 0.24, datasets 4.3)...')
    !pip install -q --force-reinstall \
        "transformers==5.5.0" "trl==0.24.0" "datasets==4.3.0" \
        "huggingface_hub" "tokenizers"
    !pip install -q peft accelerate bitsandbytes

    # Step 3: Unsloth core (2026.5.1 — latest, supports qwen3_5)
    print('Step 3: Unsloth core...')
    !pip install -q --upgrade unsloth unsloth_zoo

    # Step 4: Other deps
    print('Step 4: scipy + utilities...')
    !pip install -q --upgrade scipy
    !pip install -q sentencepiece protobuf loguru python-dotenv openai
    !pip install -q sympy chempy

    # Step 5: flash-linear-attention — Triton-based GDN kernels (always works, no CUDA build)
    # Покрывает 24/32 GDN linear-attention layers в Qwen3.5-9B
    print('Step 5: flash-linear-attention (Triton)...')
    !pip install -q flash-linear-attention 2>&1 | tail -3

    # Step 6: causal-conv1d — best-effort. Build часто падает на Colab без точного
    # nvcc/torch ABI match, но fla одной достаточно для recurrent rule (главная часть GDN).
    # Без conv1d ожидаем ~10-15 tok/s; с conv1d ~25-35 tok/s.
    print('Step 6: causal-conv1d (best-effort — fallback OK если build fails)...')
    !pip install -q causal-conv1d 2>&1 | tail -3 || echo '  conv1d build failed — fla одной хватит для GDN'

    # Step 7: Kill torchcodec — sentence_transformers ловит только (ImportError, OSError),
    # но torchcodec runtime DLL fail → RuntimeError → cascades в unsloth import.
    # Uninstall переводит fail в ImportError → caught → AudioDecoder=None → load OK.
    print('Step 7: remove torchcodec (sentence_transformers fallback нужен ImportError, не RuntimeError)...')
    !pip uninstall -y torchcodec 2>&1 | tail -1

    # Sentinel: stdlib open() — НЕ требует Path import.
    open(SENTINEL, 'w').close()
    print('=' * 60)
    print('  Install complete. RESTART RUNTIME NOW:')
    print('    Runtime → Restart session')
    print('  After restart: re-run THIS cell — install will skip, imports will load.')
    print('=' * 60)
    raise SystemExit('Restart required — re-run this cell after Runtime → Restart session.')

# ─── Post-restart imports ─────────────────────────────────
# CRITICAL: import unsloth BEFORE transformers — Unsloth patches transformers internals
# at import time. Reverse order gives "Unsloth should be imported before transformers"
# warning + missed optimizations.
import unsloth  # noqa: F401 — must precede transformers import

import sys, json, time, logging
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List

import torch
import transformers
import huggingface_hub
from unsloth import FastLanguageModel
from peft import PeftModel
from dotenv import load_dotenv

# Sanity: pinned versions должны match — иначе runtime ABI mismatch.
print(f'transformers: {transformers.__version__} | huggingface_hub: {huggingface_hub.__version__} | torch: {torch.__version__}')
assert transformers.__version__.startswith('5.5'), (
    f'Expected transformers 5.5.x (Unsloth max + qwen3_5 support). '
    f'Got {transformers.__version__}. If just installed — Runtime → Restart session.'
)
assert torch.__version__.startswith('2.10'), (
    f'Expected torch 2.10.x (Unsloth requires <2.11). Got {torch.__version__}. '
    f'If just downgraded — Runtime → Restart session.'
)

PROJECT_ROOT = Path('/content/MITS')
sys.path.insert(0, str(PROJECT_ROOT))

# Load .env from Drive (Cerebras keys + optional HF_TOKEN для приватных adapter'ов)
ENV_PATH = Path('/content/drive/MyDrive/MITS_secrets/.env')
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    print(f'Loaded env from {ENV_PATH}')
    if os.environ.get('HF_TOKEN'):
        from huggingface_hub import login as hf_login
        hf_login(token=os.environ['HF_TOKEN'])
        print('HF authenticated via .env')
    else:
        print('Warning: HF_TOKEN missing in .env — adapter download может 401 если приватные.')
else:
    print(f'No .env at {ENV_PATH}. Cerebras judge будет fail в Phase 0b.')

from training.scripts.evaluate_stage import (
    SYSTEM_PROMPT_CALC,
    extract_answer,
    check_format_compliance,
    evaluate_combined_quality,
    load_eval_dataset,
)
from training.cerebras_client import CerebrasClient

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('honest_eval')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f'Device: {DEVICE} | bf16: {torch.cuda.is_bf16_supported()}')


## Cell 3 — Decoding Configuration (зафиксирован)

Параметры применяются ОДИНАКОВО ко всем трём моделям (base, GSPO, KTO). Зафиксировано:

| Параметр | Значение | Обоснование |
|----------|----------|-------------|
| `num_predict` | 4096 | Покрывает 95-percentile thinking длин (GSPO учился с budget=2048; 4096 даёт запас на hard problems без overhead 8192). |
| `enable_thinking` | `True` | Матчит training distribution GSPO/KTO + native режим Qwen3.5-9B. False нивелировал бы RL-effect целиком — unfair. |
| `temperature` | 0.0 | Greedy для deterministic accuracy. Diversity sampling — Phase 1 (V-STaR). |
| `system_prompt` | `SYSTEM_PROMPT_CALC` | Apple-to-apple с предыдущими compare_base_vs_gspo отчётами. |

**Что это даёт для диплома**: section "Methodology — inference protocol" в одну таблицу. Альтернатива (запустить второй раз с `enable_thinking=False`) — рассматривается как _Appendix-grade ablation_, если останется compute после Phase 1-3.

In [ ]:
# Cell 4: Decoding config (filled — single-protocol run)
# Same config applied to all three models (base, GSPO, KTO).

DECODING_CONFIG = {
    # 512: drop с 2048 для overnight feasibility. На slow path (~2 tok/s без
    # causal-conv1d на Colab) 2048 → 17 min/problem; 512 → 4 min/problem.
    # Truncation risk ~10% на hard problems with long thinking, но IDENTICAL
    # cap для всех 3 моделей → apple-to-apple comparison preserved.
    'num_predict': 512,
    # True: matches GSPO/KTO training distribution (chat_template_kwargs.enable_thinking
    # =True in grpo_qwen3_5_9b_(8).ipynb). Qwen3.5-9B base also natively supports <think>.
    # False would nullify the RL effect — unfair to GSPO/KTO. Keep one consistent mode.
    'enable_thinking': True,
    # 0.0: greedy decoding for accuracy eval. Diversity sampling is for V-STaR (Phase 1).
    'temperature': 0.0,
    # Same calc system prompt used in compare_base_vs_gspo_*.json — preserves apple-to-
    # apple with prior reports for the Cerebras-only path; only inference layer changes.
    'system_prompt': SYSTEM_PROMPT_CALC,
    'rationale': (
        'Single thinking-on protocol mirrors training distribution + production deployment. '
        '2048 budget covers GSPO training distribution. Greedy decoding for deterministic accuracy.'
    ),
}

assert all(v is not None for v in DECODING_CONFIG.values()), 'Fill in DECODING_CONFIG keys'
logger.info(f'Decoding config: {DECODING_CONFIG}')

In [ ]:
# Cell 5: Load eval dataset — calc subset only (numeric / latex_boxed)
EVAL_PATH = PROJECT_ROOT / 'training/data/eval_dataset.jsonl'
all_problems = load_eval_dataset(str(EVAL_PATH))
calc_problems = [p for p in all_problems if p.get('answer_type') in ('numeric', 'latex_boxed')]
logger.info(f'Total: {len(all_problems)} | Calc subset: {len(calc_problems)}')
# Sanity
from collections import Counter
logger.info(f'By domain: {Counter(p["domain"] for p in calc_problems)}')
logger.info(f'By difficulty: {Counter(p["difficulty"] for p in calc_problems)}')

In [ ]:
# Cell 5: Model loading via Unsloth (matches training stack from grpo_qwen3.5_9b.ipynb)
BASE_MODEL_ID = 'Qwen/Qwen3.5-9B'  # matches BASE_MODEL in training notebook
MAX_SEQ_LENGTH = 4096  # covers thinking budget 2048 + completion + headroom

ADAPTERS = {
    'base': None,
    'gspo': 'Siesher/mits-qwen3-9b-gspo',
    'kto':  'Siesher/mits-qwen3-9b-kto',
}


def load_model_with_adapter(adapter_id: str | None):
    """Load Qwen3.5-9B via Unsloth FastLanguageModel, optionally apply LoRA + merge.

    Why Unsloth: Qwen3.5-9B is image-text-to-text (multimodal) с Model Class
    AutoModelForImageTextToText. AutoModelForCausalLM не работает напрямую.
    FastLanguageModel.from_pretrained абстрагирует text-only branch loading,
    skipping vision tower (saves ~2GB VRAM + matches training stack exactly).

    Adapter merge_and_unload даёт plain HF model для inference path,
    FastLanguageModel.for_inference активирует Unsloth fast kernels.
    """
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
        dtype=torch.bfloat16,
    )
    # Unsloth wraps tokenizer in some versions — unwrap if needed.
    if not hasattr(tokenizer, 'vocab_size') and hasattr(tokenizer, 'tokenizer'):
        tokenizer = tokenizer.tokenizer

    if adapter_id:
        model = PeftModel.from_pretrained(model, adapter_id)
        model = model.merge_and_unload()
        logger.info(f'Merged adapter {adapter_id}')

    FastLanguageModel.for_inference(model)
    return model, tokenizer


def generate_one(model, tokenizer, prompt: str) -> str:
    messages = [
        {'role': 'system', 'content': DECODING_CONFIG['system_prompt']},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=DECODING_CONFIG['enable_thinking'],
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=DECODING_CONFIG['num_predict'],
            do_sample=DECODING_CONFIG['temperature'] > 0,
            temperature=max(DECODING_CONFIG['temperature'], 1e-5),
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return completion


In [ ]:
# Cell 7: Eval loop (Phase 0a — fast_mode=True, programmatic correctness only)
# Phase 0b async judge — отдельным скриптом потом, когда Cerebras quota свободна.
import re

def _normalize_for_compare(s) -> str:
    """Normalize answer string for numeric/symbolic comparison.

    Accepts str | int | float | None. eval_dataset.jsonl держит numeric
    ground_truth как int/float (JSON не coerce-ит в строки), поэтому
    str(s) делается до strip().
    """
    if s is None:
        return ''
    s = str(s).strip()
    if not s:
        return ''
    if s.startswith('$') and s.endswith('$'):
        s = s[1:-1].strip()
    s = re.sub(r'\\text\{[^}]*\}', '', s)
    s = re.sub(r'\\mathrm\{[^}]*\}', '', s)
    # Russian decimal comma → dot, strip whitespace incl. nbsp
    s = s.replace(' ', '').replace(' ', '').replace(',', '.')
    # Strip trailing punctuation
    s = s.rstrip('.;,')
    return s


def is_numeric_correct(extracted, ground_truth, tolerance: float = 0.02) -> bool | None:
    """Programmatic correctness check.

    - For numeric strings: relative tolerance 2% or absolute < 0.001 if truth ≈ 0.
    - For non-numeric: case-insensitive normalized string match.
    - Returns None if can't determine (empty extracted or both unparseable).
    """
    e_norm = _normalize_for_compare(extracted)
    g_norm = _normalize_for_compare(ground_truth)
    if not e_norm or not g_norm:
        return None
    try:
        ef, gf = float(e_norm), float(g_norm)
        if abs(gf) < 1e-9:
            return abs(ef) < 0.001
        return abs(ef - gf) / abs(gf) < tolerance
    except ValueError:
        return e_norm.lower() == g_norm.lower()


def eval_stage(stage_name: str, adapter_id: str | None, problems: List[Dict],
               fast_mode: bool = True) -> Dict[str, Any]:
    """Evaluate one stage.

    Args:
        fast_mode: Phase 0a (True) — programmatic correctness only, saves raw
                   completions для Phase 0b (async Cerebras judge).
                   Phase 0 original (False) — синхронный Cerebras judge per problem
                   (упирается в rate limits, не рекомендуется).
    """
    logger.info(f'=== Stage: {stage_name} ({adapter_id or "base"}) | fast_mode={fast_mode} ===')
    model, tokenizer = load_model_with_adapter(adapter_id)
    cerebras = None if fast_mode else CerebrasClient()
    completions = []
    t0 = time.time()
    for i, p in enumerate(problems):
        t_start = time.time()
        completion = generate_one(model, tokenizer, p['prompt'])
        t_gen = time.time() - t_start
        # Per-problem timing для первых 3 — диагностика fast-path vs fallback скорости.
        if i < 3:
            n_tokens = len(tokenizer.encode(completion))
            logger.info(f'  [{i+1}/{len(problems)}] gen={t_gen:.1f}s | tokens={n_tokens} | tok/s={n_tokens/max(t_gen,0.01):.1f}')
        extracted = extract_answer(completion)
        fmt = check_format_compliance(completion)
        visible = completion.split('</think>')[-1].strip() if '</think>' in completion else completion

        if fast_mode:
            correct = is_numeric_correct(extracted, p['ground_truth'])
            socratic, no_leak = None, None
        else:
            judge = evaluate_combined_quality(p['prompt'], visible, p['ground_truth'], cerebras)
            correct = judge.get('is_correct')
            socratic = judge.get('socratic_score')
            no_leak = judge.get('no_answer_leak')

        completions.append({
            'idx': i, 'domain': p['domain'], 'difficulty': p['difficulty'],
            'truth': p['ground_truth'], 'extracted': extracted,
            'correct': correct,
            'socratic_score': socratic,
            'no_answer_leak': no_leak,
            'completion_text': visible,  # saved для Phase 0b async judge
            **fmt,
        })

        log_every = 5 if fast_mode else 10
        if (i + 1) % log_every == 0:
            elapsed = time.time() - t0
            valid = [c for c in completions if c['correct'] is not None]
            acc = sum(1 for c in valid if c['correct']) / max(len(valid), 1)
            logger.info(f'  {i+1}/{len(problems)} | acc={acc:.3f} ({len(valid)} judged) | {elapsed/60:.1f} min')

    del model; torch.cuda.empty_cache()
    valid = [c for c in completions if c['correct'] is not None]
    accuracy = sum(1 for c in valid if c['correct']) / max(len(valid), 1)
    socratic_vals = [c['socratic_score'] for c in completions if c.get('socratic_score') is not None]
    leak_vals = [c['no_answer_leak'] for c in completions if c.get('no_answer_leak') is not None]
    return {
        'stage': stage_name, 'adapter': adapter_id, 'n': len(completions),
        'n_judged': len(valid),
        'accuracy': accuracy,
        'avg_socratic': (sum(socratic_vals) / len(socratic_vals)) if socratic_vals else None,
        'leak_rate': (sum(1 for v in leak_vals if v < 2) / len(leak_vals)) if leak_vals else None,
        'mode': 'fast_programmatic' if fast_mode else 'full_cerebras_judge',
        'completions': completions,
    }

In [ ]:
# Cell 7.5: Stratified 30-sample subset для overnight feasibility.
# RATIONALE: после 8+ infra fix attempts установили что Colab + Qwen3.5-9B
# fast path требует causal-conv1d, который не собирается на cu128/nvcc 12.8 mix.
# Slow path даёт ~2 tok/s, что для full 143 × 3 stages = 120h. Невозможно.
#
# Solution: 30 stratified (10 per difficulty) × num_predict=512 (Cell 3 patched)
# = ~6-7h overnight. Identical sampling/budget для всех 3 моделей.
#
# Methodological note для диплома: "Eval performed on 30-problem stratified
# subset (10 per difficulty: easy/medium/hard) due to unresolved fla/conv1d
# ABI incompatibility on Colab. Stratification preserves difficulty distribution
# of full 143-problem benchmark; num_predict=512 caps generation для compute
# feasibility. Apple-to-apple comparison preserved across base/GSPO/KTO."
import random
from collections import defaultdict

random.seed(42)  # reproducible stratification

by_diff = defaultdict(list)
for p in calc_problems:
    by_diff[p.get('difficulty', 'medium')].append(p)

logger.info(f'Calc problems by difficulty: {[(k, len(v)) for k, v in by_diff.items()]}')

stratified = []
N_PER_DIFF = 10
for diff in ['easy', 'medium', 'hard']:
    pool = by_diff.get(diff, [])
    n_take = min(N_PER_DIFF, len(pool))
    sampled = random.sample(pool, n_take) if n_take > 0 else []
    stratified.extend(sampled)
    logger.info(f'  {diff}: sampled {n_take}/{len(pool)}')

logger.info(f'Stratified subset: {len(stratified)} problems total')

# Rebind для Cell 8 (run cell uses calc_problems variable)
calc_problems_full = calc_problems  # backup в случае нужно вернуться к full
calc_problems = stratified
logger.info(f'calc_problems rebound to subset (use calc_problems_full для full 143)')


In [ ]:
# Cell 8: Run Phase 0a (fast_mode=True — programmatic correctness, no Cerebras).
# Phase 0b (async Cerebras judge over saved completions) — отдельный скрипт потом.
results = {}
for stage_name, adapter_id in ADAPTERS.items():
    results[stage_name] = eval_stage(stage_name, adapter_id, calc_problems, fast_mode=True)

report = {
    'protocol': 'honest_full_precision_phase0a',
    'mode': 'fast_programmatic',
    'note': (
        'Phase 0a: accuracy via programmatic numeric/string match (extract_answer vs '
        'ground_truth, 2% tolerance). socratic_score / leak_rate deferred to Phase 0b '
        '(async Cerebras judge over saved completions).'
    ),
    'timestamp': datetime.utcnow().isoformat(),
    'decoding_config': {k: v for k, v in DECODING_CONFIG.items() if k != 'system_prompt'},
    'system_prompt_hash': hash(DECODING_CONFIG['system_prompt']),
    'n_problems': len(calc_problems),
    'base': {k: v for k, v in results['base'].items() if k != 'completions'},
    'gspo': {k: v for k, v in results['gspo'].items() if k != 'completions'},
    'kto':  {k: v for k, v in results['kto'].items()  if k != 'completions'},
    'completions': {stage: r['completions'] for stage, r in results.items()},  # raw text сохранён
}

# Save to repo (so it can be committed) + Drive mirror (so it survives Colab teardown)
date_tag = datetime.utcnow().strftime('%Y%m%d')
out_repo  = PROJECT_ROOT / f'evaluation/reports/honest_full_precision_phase0a_{date_tag}.json'
out_drive = Path(f'/content/drive/MyDrive/MITS_secrets/honest_full_precision_phase0a_{date_tag}.json')
out_repo.parent.mkdir(parents=True, exist_ok=True)
out_repo.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
if out_drive.parent.exists():
    out_drive.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    logger.info(f'Saved (Drive mirror): {out_drive}')
logger.info(f'Saved (repo): {out_repo}')

print('\n=== Phase 0a — Honest accuracy (full-precision bf16, identical decoding, programmatic correctness) ===')
for stage in ['base', 'gspo', 'kto']:
    r = results[stage]
    print(f"{stage:5s}  acc={r['accuracy']:.3f} ({r['n_judged']}/{r['n']} judged)")
print('\nNote: socratic_score / leak_rate — Phase 0b (run scripts/score_phase0b_async.py later).')